# 3. You cannot have every fairness definition at once

**Claim under test:** the output of a data pipeline can be "cleaned, balanced,
compliant", as if fairness were a property you finish installing.

It is not. When base rates genuinely differ between groups, calibration, equal
error rates and demographic parity are **mathematically incompatible**. Choosing
one is a value judgement, not a technical step, and it happens after the data
work, at the threshold.

- Kleinberg, Mullainathan & Raghavan (2016) and Chouldechova (2017) - the two
  impossibility results (`papers/`)
- Narayanan, *21 Fairness Definitions and Their Politics* (2018) - why the choice
  is political (`papers/narayanan-2018-21-fairness-definitions.pdf`)
- Hardt, Price & Srebro (2016) - equalised odds

In [1]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.calibration import calibration_curve

RNG = np.random.default_rng(3)
N = 20000

group_b = RNG.random(N) < 0.35
x = RNG.normal(np.where(group_b, -0.45, 0.15), 1.0, N)     # risk signal
y = RNG.binomial(1, 1 / (1 + np.exp(-(1.4 * x + 0.4))))    # true outcome

X = pd.DataFrame({'x': x})
p = LogisticRegression().fit(X, y).predict_proba(X)[:, 1]

a, b = ~group_b, group_b
print(f'base rate   A={y[a].mean():.1%}   B={y[b].mean():.1%}   <- genuinely different')

base rate   A=60.7%   B=46.9%   <- genuinely different


The base rates differ. That single fact is what makes the rest impossible. Note
the model never sees group membership.

In [2]:
def report(decision, title):
    print(title)
    stats = {}
    for name, mask in (('A', a), ('B', b)):
        yy, dd = y[mask], decision[mask]
        stats[name] = dict(sel=dd.mean(), tpr=dd[yy == 1].mean(),
                           fpr=dd[yy == 0].mean(), ppv=yy[dd].mean())
        s = stats[name]
        print(f"   {name}: selected={s['sel']:.1%}  TPR={s['tpr']:.3f}  "
              f"FPR={s['fpr']:.3f}  PPV={s['ppv']:.3f}")
    print(f"   demographic parity ratio = {stats['B']['sel'] / stats['A']['sel']:.2f}")
    print(f"   TPR gap = {abs(stats['A']['tpr'] - stats['B']['tpr']):.3f}   "
          f"PPV gap = {abs(stats['A']['ppv'] - stats['B']['ppv']):.3f}\n")


report(p >= 0.5, 'ONE THRESHOLD FOR EVERYONE (group-blind, calibrated)')

t_a = np.quantile(p[a], 1 - 0.35)
t_b = np.quantile(p[b], 1 - 0.35)
report(np.where(b, p >= t_b, p >= t_a), 'FORCED DEMOGRAPHIC PARITY (35% accepted in each group)')
print(f'   thresholds: A={t_a:.3f}  B={t_b:.3f}  '
      f'-> the same score is treated differently depending on group')

ONE THRESHOLD FOR EVERYONE (group-blind, calibrated)
   A: selected=66.8%  TPR=0.837  FPR=0.407  PPV=0.761
   B: selected=45.1%  TPR=0.697  FPR=0.234  PPV=0.725
   demographic parity ratio = 0.68
   TPR gap = 0.140   PPV gap = 0.036

FORCED DEMOGRAPHIC PARITY (35% accepted in each group)
   A: selected=35.0%  TPR=0.504  FPR=0.111  PPV=0.875
   B: selected=35.0%  TPR=0.579  FPR=0.148  PPV=0.776
   demographic parity ratio = 1.00
   TPR gap = 0.074   PPV gap = 0.099

   thresholds: A=0.764  B=0.589  -> the same score is treated differently depending on group


## Is the model calibrated?

Calibration means a predicted 0.7 corresponds to a real 70%, within each group.
It is the property a lender most wants, and the group-blind model has it.

In [3]:
for name, mask in (('A', a), ('B', b)):
    frac, mean_pred = calibration_curve(y[mask], p[mask], n_bins=5, strategy='quantile')
    print(f'  {name}: predicted {np.round(mean_pred, 3)}')
    print(f'     actual    {np.round(frac, 3)}')

  A: predicted [0.22  0.465 0.648 0.797 0.923]
     actual    [0.216 0.452 0.649 0.794 0.925]
  B: predicted [0.109 0.277 0.455 0.636 0.838]
     actual    [0.115 0.276 0.444 0.663 0.849]


## Read the numbers

The group-blind model is well calibrated in both groups: predicted and actual
line up. It also fails the four-fifths rule, at a parity ratio of 0.68.

Forcing parity fixes that ratio to 1.00 and breaks two other things. The PPV gap
opens up, so an accepted applicant from one group is meaningfully less likely to
repay than an accepted applicant from the other. And the two thresholds differ,
which means **the same score produces a different decision depending on group
membership**. In lending that is a legal problem, not just a technical one.

There is no arrangement of this data that removes the tension. It follows from
the base rates differing, and it is a theorem, not a limitation of the model.

**What this does to the flowchart.** The output box says "cleaned, balanced,
compliant training data ready for modeling". Nothing upstream of the threshold
can deliver that, because the conflict appears at the threshold and is
irreducible. What a defensible pipeline produces is not compliant data. It is a
documented, deliberate choice of which fairness definition you are optimising,
and the evidence for why.